# ArthJAX — GPU Macro ABM + World Model

**Before running:** Kaggle settings → **Accelerator: GPU**, **Internet: On**.

Installs [ArthJAX](https://github.com/VARUN3WARE/ArthJAX) from GitHub and runs:
1. 600-step JIT simulation
2. Macro charts
3. Neural world model training + real vs learned comparison

Outputs save to `/kaggle/working/`.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/VARUN3WARE/ArthJAX.git",
])

In [ ]:
import os
import time

import jax
import jax.numpy as jnp
import numpy as np

from arthjax import EconomyConfig, __version__, init_state, simulate
from arthjax.simulation.step import make_step_jit

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"ArthJAX v{__version__}")
print(f"JAX {jax.__version__} | devices: {jax.devices()}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
cfg = EconomyConfig(default_num_steps=600, default_seed=42)
key = jax.random.PRNGKey(cfg.default_seed)
key, init_key = jax.random.split(key)

state = init_state(cfg, init_key)
step_jit = make_step_jit(cfg)

t0 = time.time()
final_state, metrics_history = simulate(
    state, cfg.default_num_steps, key, cfg=cfg, step_jit=step_jit
)
elapsed = time.time() - t0
metrics_np = jax.tree.map(np.array, metrics_history)

print(f"Simulation: {elapsed:.2f}s ({cfg.default_num_steps / elapsed:.1f} steps/sec)")
print(f"GDP: ${metrics_np['gdp'][-1]:.0f}")
print(f"Inflation: {metrics_np['inflation'][-1] * 100:.2f}%")
print(f"Unemployment: {metrics_np['unemployment'][-1] * 100:.2f}%")
print(f"Sentiment: {metrics_np['sentiment'][-1]:.3f}")

In [ ]:
from arthjax.viz.plots import save_all_plots

save_all_plots(metrics_np, OUTPUT_DIR, cfg)
print("Saved: macro_evolution_v2.png, boom_bust_v2.png, linkedin_hero.png")

In [ ]:
import jax.random as jr

from arthjax.world_model.rollout import (
    collect_real_trajectory,
    compare_rollouts,
    rollout_learned,
)
from arthjax.world_model.train import train_world_model
from arthjax.viz.plots import plot_real_vs_learned, plot_world_model_loss

wm_key = jr.PRNGKey(100)
wm_key, init_key = jr.split(wm_key)
wm_state = init_state(cfg, init_key)

t0 = time.time()
params, normalizer, losses = train_world_model(wm_state, step_jit, cfg, wm_key)
print(f"Training: {time.time() - t0:.1f}s | final loss={losses[-1]:.6f}")

eval_key = jr.PRNGKey(200)
real_traj = collect_real_trajectory(wm_state, step_jit, cfg, eval_key)
learned_traj = rollout_learned(
    params, real_traj[0], normalizer, cfg, num_steps=len(real_traj) - 1
)
wm_metrics = compare_rollouts(real_traj, learned_traj)

print(f"Macro MAE: {wm_metrics['macro_mae']:.6f} ({wm_metrics['macro_relative_error_pct']:.2f}% relative)")

plot_world_model_loss(losses, os.path.join(OUTPUT_DIR, "world_model_loss_v2.png"))
plot_real_vs_learned(
    real_traj, learned_traj, os.path.join(OUTPUT_DIR, "real_vs_learned_v2.png")
)
print("Saved: world_model_loss_v2.png, real_vs_learned_v2.png")

## Done

Charts are in `/kaggle/working/`. Download from the **Output** tab.

- Repo: https://github.com/VARUN3WARE/ArthJAX
- Local CLI: `python scripts/run_simulation.py --steps 600 --plot`